In [25]:

# ================================================================
# EXPERIMENT 23 — E21 TARGETED FINE-TUNING
# Same ResNet34 U-Net, same weighted CE + Dice loss
# Conservative learning rate, no focal loss
# ================================================================

import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ---------------- CONFIGURATION ----------------

SEED = 42
ROOT = "/home/jovyan/vault/LULC_Project"

E21_PATH = (
    ROOT +
    "/results/exp21_fresh_422_resnet34_unet/"
    "checkpoints/best_exp21_resnet34_unet.pth"
)

OUT = ROOT + "/results/exp23_e21_targeted_finetune"
CKPT_DIR = OUT + "/checkpoints"

os.makedirs(CKPT_DIR, exist_ok=True)

BEST_PATH = CKPT_DIR + "/best_exp23.pth"
LAST_PATH = CKPT_DIR + "/last_exp23.pth"

MAX_EPOCHS = 100
PATIENCE = 45

LEARNING_RATE = 3e-6
WEIGHT_DECAY = 5e-5
BATCH_SIZE = 8

CLASS_NAMES = [
    "Tree",
    "Shrub/Grass/Moss",
    "Cropland",
    "Built-up",
    "Bare/Snow/Ice",
    "Water/Wetland/Mangrove"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 75)
print("EXPERIMENT 23 — E21 TARGETED FINE-TUNING")
print("=" * 75)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------- CHECK EXISTING DEFINITIONS ----------------

required = [
    "LULCDataset",
    "ResNet34UNet",
    "combined_loss",
    "evaluate",
    "train_loader",
    "val_loader"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing E21 notebook definitions: " + str(missing) +
        ". Run the relevant E21 setup cells first."
    )

if not os.path.exists(E21_PATH):
    raise FileNotFoundError(E21_PATH)

# ---------------- UN-AUGMENTED TRAIN METRICS ----------------

train_metric_dataset = LULCDataset("train", augment=False)

train_metric_loader = DataLoader(
    train_metric_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# ---------------- LOAD E21 CHECKPOINT ----------------

model = ResNet34UNet(
    in_channels=48,
    num_classes=6
).to(DEVICE)

checkpoint = torch.load(E21_PATH, map_location=DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])

print("\nLoaded E21 checkpoint")
print("Original best epoch:", checkpoint["epoch"])
print("Original validation mIoU:",
      round(checkpoint["val_miou"] * 100, 3), "%")

# ---------------- OPTIMIZER ----------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=3e-7
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# ---------------- TRAINING FUNCTION ----------------

def train_e23_epoch():

    model.train()
    total_loss = 0.0

    for images, masks in train_loader:

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(images)
            loss, _, _ = combined_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(train_loader)


# ---------------- TRAINING LOOP ----------------

history = []
best_val_miou = -1.0
best_epoch = 0
no_improvement = 0

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    train_loss = train_e23_epoch()

    # Training metrics are calculated without augmentation.
    train_metrics = evaluate(train_metric_loader)
    val_metrics = evaluate(val_loader)

    val_miou = float(val_metrics["miou"])

    improved = val_miou > best_val_miou

    if improved:

        best_val_miou = val_miou
        best_epoch = epoch
        no_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_miou": val_miou,
                "architecture": "ResNet34UNet",
                "experiment": "E23"
            },
            BEST_PATH
        )

    else:
        no_improvement += 1

    scheduler.step(val_miou)

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_miou": val_miou
        },
        LAST_PATH
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_metrics["oa"],
        "train_miou": train_metrics["miou"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["oa"],
        "val_miou": val_metrics["miou"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": optimizer.param_groups[0]["lr"]
    }

    history.append(row)

    pd.DataFrame(history).to_csv(
        OUT + "/training_history.csv",
        index=False
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train OA: {train_metrics['oa']*100:.2f}% | "
        f"Val OA: {val_metrics['oa']*100:.2f}% | "
        f"Train mIoU: {train_metrics['miou']*100:.2f}% | "
        f"Val mIoU: {val_metrics['miou']*100:.2f}% | "
        f"Train F1: {train_metrics['macro_f1']*100:.2f}% | "
        f"Val F1: {val_metrics['macro_f1']*100:.2f}% | "
        f"Loss: {train_loss:.4f} | "
        f"{time.time()-epoch_start:.1f}s "
        f"{'<BEST>' if improved else ''}"
    )

    if no_improvement >= PATIENCE:
        print("\nEarly stopping.")
        break

# ---------------- FINAL BEST CHECKPOINT ----------------

best = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(best["model_state_dict"])

final_train = evaluate(train_metric_loader)
final_val = evaluate(val_loader)

print("\n" + "=" * 75)
print("EXPERIMENT 23 — FINAL BEST RESULTS")
print("=" * 75)

print("Best epoch:", best["epoch"])

print(f"Train Accuracy: {final_train['oa']*100:.3f}%")
print(f"Val Accuracy:   {final_val['oa']*100:.3f}%")

print(f"Train mIoU:     {final_train['miou']*100:.3f}%")
print(f"Val mIoU:       {final_val['miou']*100:.3f}%")

print(f"Train Macro F1: {final_train['macro_f1']*100:.3f}%")
print(f"Val Macro F1:   {final_val['macro_f1']*100:.3f}%")

print("\nVALIDATION PER-CLASS METRICS")

for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name:24s} "
        f"IoU: {final_val['iou'][i]*100:7.3f}% "
        f"F1: {final_val['f1'][i]*100:7.3f}%"
    )

results = {
    "experiment": "E23",
    "best_epoch": int(best["epoch"]),
    "train_accuracy": float(final_train["oa"]),
    "val_accuracy": float(final_val["oa"]),
    "train_miou": float(final_train["miou"]),
    "val_miou": float(final_val["miou"]),
    "train_macro_f1": float(final_train["macro_f1"]),
    "val_macro_f1": float(final_val["macro_f1"]),
    "validation_iou": [
        float(x) for x in final_val["iou"]
    ],
    "validation_f1": [
        float(x) for x in final_val["f1"]
    ]
}

with open(OUT + "/final_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nElapsed minutes:", round((time.time()-start_time)/60, 2))
print("Best checkpoint:", BEST_PATH)
print("History:", OUT + "/training_history.csv")
print("\nE23 COMPLETE — TEST SET NOT USED.")


EXPERIMENT 23 — E21 TARGETED FINE-TUNING
Device: cuda
GPU: NVIDIA RTX A6000
train: 422 tiles

Loaded E21 checkpoint
Original best epoch: 41
Original validation mIoU: 49.39 %
Epoch 01/100 | Train OA: 78.50% | Val OA: 70.54% | Train mIoU: 66.21% | Val mIoU: 49.00% | Train F1: 78.91% | Val F1: 61.77% | Loss: 0.5861 | 34.3s <BEST>
Epoch 02/100 | Train OA: 78.59% | Val OA: 70.99% | Train mIoU: 66.46% | Val mIoU: 49.42% | Train F1: 79.18% | Val F1: 62.30% | Loss: 0.5578 | 27.8s <BEST>
Epoch 03/100 | Train OA: 78.63% | Val OA: 70.78% | Train mIoU: 66.42% | Val mIoU: 49.29% | Train F1: 79.15% | Val F1: 62.07% | Loss: 0.5910 | 23.7s 
Epoch 04/100 | Train OA: 78.71% | Val OA: 70.05% | Train mIoU: 66.59% | Val mIoU: 48.51% | Train F1: 79.24% | Val F1: 61.47% | Loss: 0.5917 | 22.4s 
Epoch 05/100 | Train OA: 78.49% | Val OA: 70.54% | Train mIoU: 66.14% | Val mIoU: 48.55% | Train F1: 78.89% | Val F1: 61.44% | Loss: 0.6003 | 24.0s 
Epoch 06/100 | Train OA: 78.92% | Val OA: 70.11% | Train mIoU: 66.95%